# Tutorial: a beam with an energy spread

Real beams have a spread of energies and directions. In pysrim2 each beam parameter can be a
**distribution**. Here the energy of boron ions is 20 keV with a Gaussian spread of 1 keV.

Two modes run the same beam:

- **exact** (the default) samples every ion, writes them to SRIM's `TRIM.DAT`, and SRIM runs
  exactly those ions;
- **binned** cuts the energy into bins, runs SRIM once per bin, and gives each bin its share of
  the ions.

The run needs SRIM in its default location (`pysrim2 install`) and the plots need matplotlib (the
`plot` extra of pysrim2). The ion count is small so that the notebook runs quickly.

In [ ]:
from srim import TRIM, Ion, Layer, Target
from srim.analysis import implant_summary
from srim.beam import Gaussian, plot_beam, run_beam, sampled_ions
from srim.spec import RunSpec

## The base run

`TRIM(...).to_spec()` gives the run spec as a plain mapping. Its energy is replaced by a
Gaussian distribution with mean 20 keV and standard deviation 1 keV (`Gaussian(20.0, 1.0)`).
`ranges=1` writes `RANGE_3D.txt`, which `implant_summary` reads.

In [ ]:
target = Target([Layer.from_formula("Si", density=2.33, width=10000.0)])  # 1000 nm
data = TRIM(
    target,
    Ion("B", energy=20e3),  # eV; replaced by the distribution below
    number_ions=200,
    calculation=1,  # quick Kinchin-Pease
    ranges=1,
    random_seed=12345,
).to_spec().to_dict()
data["ion"]["energy_keV"] = Gaussian(20.0, 1.0).to_dict()
exact_spec = RunSpec.from_dict(data)

## Exact mode

`run_beam` returns the results of the run. `sampled_ions` reads the ions SRIM ran back from
`TRIM.DAT`: their energies follow the Gaussian. `plot_beam` draws the sampled energies as a
histogram with the defined density over them. The results folder `beam-exact` is kept.

In [ ]:
with run_beam(exact_spec, output_dir="beam-exact") as exact:
    ions = sampled_ions(exact)
    summary = implant_summary(exact.range3d.df)
    ax = plot_beam(exact, "energy_keV")

print(ions["energy_keV"].describe())
print(f"mean depth {summary['mean_depth_A']:.0f} A, straggling {summary['straggle_depth_A']:.0f} A")

## Binned mode

The same beam with `beam: {mode: binned}`. Four energy bins of equal probability are cut out of
the Gaussian. Each bin is one SRIM run at the bin's mean energy, and the 200 ions are split over
the bins. `beam_bins` lists the bins and their ion counts. The results are merged, so the
per-ion tables have a `bin` column.

In [ ]:
data["beam"] = {"mode": "binned", "bins": {"energy_keV": 4}}
binned_spec = RunSpec.from_dict(data)

with run_beam(binned_spec, output_dir="beam-binned") as binned:
    print(binned.beam_bins)
    summary_binned = implant_summary(binned.range3d.df)
    ax = plot_beam(binned, "energy_keV")

print(f"mean depth {summary_binned['mean_depth_A']:.0f} A, straggling {summary_binned['straggle_depth_A']:.0f} A")

Both modes give mean depths of the same size. Exact mode keeps every sampled ion; binned mode
replaces the energies inside each bin by the bin's mean. More bins bring the two closer, at the
cost of more SRIM runs. With 200 ions the mean depth changes by several per cent from seed to
seed, so compare the modes with more ions or several seeds. See `docs/beam.md` for all distribution shapes, histogram files, Python
density functions and the limits of each mode.